<a href="https://colab.research.google.com/github/yadavrishikesh/MA-563-Statistical-Computation-and-Simulation-/blob/main/code/module1/MA563_Lab_Exam_Extended_SOLUTIONS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## MA 563 - Statistical Computation and Simulation
### Lab Examination — Module 1

<!-- Instructor: Dr. Rishikesh Yadav  |  TA: Vedant Vibhor -->

---

**Read before you start.**

1. There are **two questions** with **seven cells to complete**, marked `YOUR CODE HERE`. Question 2 uses the sampler you build in Question 1, so work in order.
2. Every incomplete cell raises `NotImplementedError` until you fill it in. Make sure to remove it when you've coded the solve.
3. All plots, summary statistics and diagnostic tests are **already written**. Do not edit those cells — they are there so you can see whether your code is right.
4. **Do not change `SEED`.**

| Question | Parts | Marks |
|:--|:--|:--|
| 1.  Cauchy by inverse transform | (a) CDF, (b) inverse CDF, (c) sampler | -- |
| 2. Normal from Cauchy by acceptance–rejection | (a) ratio, (b) constant $c$, (c) acceptance test, (d) the sampler | -- |


In [ ]:
# ==================================================================
# SETUP -- run this cell first.  Do not edit.
# ==================================================================
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    "figure.figsize": (10, 3.6), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})

BLUE, ORANGE, GREEN, RED = "#003366", "#ED7D31", "#2E7D32", "#C62828"

SEED = 2026        # fixed for the whole class -- DO NOT CHANGE
N    = 100_000     # sample size used in both questions


def fresh_rng():
    """Return a generator restarted from SEED.

    Each question calls this at the top of its driver cell, so your numbers do
    not depend on how many times you happened to re-run an earlier cell.  This
    is what makes your output reproducible at the viva.
    """
    return np.random.default_rng(SEED)


print("Setup complete.  numpy", np.__version__, " | SEED =", SEED)

In [ ]:
# ==================================================================
# Validator (the same one from Lab 1).  Do not edit.
# ==================================================================

def check_continuous(samples, dist, name, xlim=None, bins=60):
    """Histogram vs true pdf, Q-Q plot, and a Kolmogorov-Smirnov test.

    `dist` is a frozen scipy distribution -- an INDEPENDENT implementation of
    the same target, so agreement is real evidence that your sampler is right.
    """
    lo, hi = xlim if xlim is not None else (np.min(samples), np.max(samples))
    grid = np.linspace(lo, hi, 400)

    fig, ax = plt.subplots(1, 2, figsize=(10, 3.3))

    ax[0].hist(samples, bins=bins, range=(lo, hi), density=True,
               color=BLUE, edgecolor="white", alpha=0.85, label="your samples")
    ax[0].plot(grid, dist.pdf(grid), color=ORANGE, lw=2.2, label="true pdf")
    ax[0].set_title(f"{name}: density check")
    ax[0].set_xlabel("x"); ax[0].legend(fontsize=8)

    p = (np.arange(1, 201) - 0.5) / 200
    ax[1].plot(dist.ppf(p), np.quantile(samples, p), ".", ms=4, color=BLUE)
    lim = [max(lo, dist.ppf(0.001)), min(hi, dist.ppf(0.999))]
    ax[1].plot(lim, lim, "--", color=ORANGE, lw=1.5)
    ax[1].set_title(f"{name}: Q-Q plot")
    ax[1].set_xlabel("theoretical quantile"); ax[1].set_ylabel("sample quantile")
    ax[1].set_xlim(lim); ax[1].set_ylim(lim)

    plt.tight_layout(); plt.show()

print("validator ready")

---
## Question 1 - The standard Cauchy by inverse transform  *(30 marks)*

The standard Cauchy distribution has density

$$f(x) = \frac{1}{\pi(1+x^2)}, \qquad x \in \mathbb{R}.$$

**(a)** Integrate $f$ to obtain the CDF $F$, and implement it.

**(b)** $F$ is continuous and strictly increasing on all of $\mathbb{R}$, so the
probability integral transform applies: if $U \sim \mathrm{Uniform}(0,1)$ then
$X = F^{-1}(U)$ has CDF $F$. Set $u = F(x)$, solve for $x$, and implement $F^{-1}$.

**(c)** Write the sampler itself - draw the uniforms and push them through $F^{-1}$.

In [ ]:
# ==================================================================
# Q1 (a), (b), (c)  --  SOLUTION
# ==================================================================

def cauchy_cdf(x):
    """F(x) = 1/2 + arctan(x)/pi.

    Since  d/dt [arctan(t)/pi] = 1/(pi(1+t^2)) = f(t), and arctan(-inf) = -pi/2,
    integrating f from -infinity to x gives 1/2 + arctan(x)/pi.
    """
    return 0.5 + np.arctan(x) / np.pi


def cauchy_ppf(u):
    """Inverse of the above.

    u = 1/2 + arctan(x)/pi
      => arctan(x) = pi*(u - 1/2)
      => x         = tan(pi*(u - 1/2))
    """
    return np.tan(np.pi * (u - 0.5))


def sample_cauchy(k, rng):
    """U ~ Uniform(0,1), then X = F^{-1}(U)."""
    return cauchy_ppf(rng.random(k))

In [ ]:
# ==================================================================
# Q1 self-check -- given.  Do not edit.
#
# Two consistency tests that need no simulation at all:
#   1. F(F^{-1}(u)) should return u exactly (to floating-point precision).
#   2. Your CDF should agree with scipy's, which is an independent
#      implementation of the same function.
# Both errors should be at the 1e-15 level.  Anything larger is a bug in (a)
# or (b), and there is no point running the sampler until it is fixed.
# ==================================================================
u_test = np.linspace(0.001, 0.999, 5001)
x_test = np.linspace(-30, 30, 5001)

err_roundtrip = np.abs(cauchy_cdf(cauchy_ppf(u_test)) - u_test).max()
err_vs_scipy  = np.abs(cauchy_cdf(x_test) - stats.cauchy().cdf(x_test)).max()

print(f"max |F(F^-1(u)) - u|        = {err_roundtrip:.3e}")
print(f"max |your F(x) - scipy F(x)| = {err_vs_scipy:.3e}")
print()
print("PASS" if max(err_roundtrip, err_vs_scipy) < 1e-12
      else "FAIL -- fix (a)/(b) before going on")

In [ ]:
# ==================================================================
# Q1 diagnostics -- given.  Do not edit.
# ==================================================================
rng = fresh_rng()
x_cauchy = sample_cauchy(N, rng)

# The support is all of R, so the picture is clipped to (-8, 8) -- but the KS
# test below uses every sample, tails included.
check_continuous(x_cauchy, stats.cauchy(), "Standard Cauchy", xlim=(-8, 8))

---
## Question 2 - Normal $(0,1)$ from your Cauchy, by acceptance–rejection  

The Normal has no closed-form $F^{-1}$, so the inverse transform of Question 1 is
unavailable. Instead we use the Cauchy sampler you just built as the **envelope**.

$$\text{target}\quad f(x) = \frac{1}{\sqrt{2\pi}}e^{-x^2/2},
\qquad\qquad
\text{envelope}\quad g(x) = \frac{1}{\pi(1+x^2)}.$$

The Cauchy is the right choice because its tails are **heavier** than the Normal's, so
the ratio $f/g$ stays bounded.

**Algorithm.** Draw $Y \sim g$ and $U \sim \mathrm{Uniform}(0,1)$ independently. Accept
$X = Y$ if

$$U \le \frac{f(Y)}{c\,g(Y)},$$

otherwise discard the pair and repeat. Correctness requires
$c \ge \sup_x f(x)/g(x)$; the acceptance probability is then $1/c$, and the number of
proposals per accepted draw is $\mathrm{Geometric}(1/c)$ with mean $c$.

**Your task.**

**(a)** Simplify $f(x)/g(x)$ algebraically and implement it as a single expression.

**(b)** Maximise that ratio by hand and set `C` to the supremum.

**(c)** Implement the acceptance test.

**(d)** Write the sampler: keep proposing until you have `n` acceptances.

*A warning about (b).* Choosing $c$ **too small** breaks correctness - the accepted draws
are then not Normal at all, and the sampler will warn you. Choosing $c$ too large costs
nothing but time, and **nothing in the output will complain**. Only the correct $c$ makes
the empirical acceptance rate match $1/c$ *and* with the minimum proposal count. (Will be checked)

In [ ]:
# ==================================================================
# Q2 (a), (b)  --  SOLUTION
# ==================================================================

# ---- given: the two densities -------------------------------------
f_normal = lambda x: np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)   # target   f
g_cauchy = lambda x: 1.0 / (np.pi * (1.0 + x**2))             # envelope g


def ratio(x):
    """f(x)/g(x) = sqrt(pi/2) * (1 + x^2) * exp(-x^2/2).

    [exp(-x^2/2)/sqrt(2 pi)] * [pi(1+x^2)]
        = (pi/sqrt(2 pi)) * (1+x^2) * exp(-x^2/2)
        = sqrt(pi/2)      * (1+x^2) * exp(-x^2/2)
    """
    return np.sqrt(np.pi / 2) * (1 + x**2) * np.exp(-x**2 / 2)


# d/dx [ (1+x^2) exp(-x^2/2) ] = x(1 - x^2) exp(-x^2/2), which vanishes at
# x = 0, +1, -1.  The value at x = 0 is 1; at x = +-1 it is 2*exp(-1/2) > 1,
# so the maximum sits at x = +-1 and
#
#     c = sqrt(pi/2) * 2 * exp(-1/2)  ~= 1.5203
C = math.sqrt(math.pi / 2) * 2 * math.exp(-0.5)

In [ ]:
# ==================================================================
# Q2 (a) self-check -- given.  Do not edit.
#
# Your simplified ratio must agree with the raw quotient everywhere.
# ==================================================================
xg = np.linspace(-10, 10, 20_001)
err_ratio = np.abs(ratio(xg) - f_normal(xg) / g_cauchy(xg)).max()

print(f"max |your ratio(x) - f(x)/g(x)| = {err_ratio:.3e}")
print("PASS" if err_ratio < 1e-12 else "FAIL -- your algebra in (a) is wrong")
print(f"\nC = {C:.6f},  so the method should accept about {1/C:.2%} of proposals.")

In [ ]:
# ==================================================================
# Q2 (c)  --  SOLUTION
# ==================================================================

def accept_mask(y, u, C):
    """Accept when U <= f(Y) / (C * g(Y)) = ratio(Y) / C."""
    return u <= ratio(y) / C

In [ ]:
# ==================================================================
# Given helper for Q2 (d).  Do not edit.
#
# Turning a long run of proposals into "the first n accepted draws, and how
# many proposals that cost" is tricky index arithmetic with no statistical
# content, so it is written for you.  Call it at the end of your sampler.
# ==================================================================

def acceptance_tracker(y, keep, n):
    """Extract the first n accepted proposals and the efficiency diagnostics.

    Parameters
    ----------
    y    : 1-D array of ALL proposals generated, in order
    keep : boolean array of the same length, True where accepted
    n    : how many accepted draws are wanted

    Returns
    -------
    samples : array of length exactly n
    info    : dict with n_proposals, acceptance_rate, trials
    """
    idx = np.flatnonzero(keep)[:n]                  # positions of the acceptances
    if len(idx) < n:
        raise RuntimeError(f"only {len(idx)} acceptances but {n} were requested -- "
                           f"your loop stopped too early")
    n_prop = int(idx[-1]) + 1                       # proposals consumed to get them
    trials = np.diff(np.concatenate(([-1], idx)))   # proposals used per acceptance
    return y[idx], {"n_proposals": n_prop,
                    "acceptance_rate": n / n_prop,
                    "trials": trials}


print("helper ready")

In [ ]:
# ==================================================================
# Q2 (d)  --  SOLUTION
# ==================================================================

def normal_from_cauchy(n, C, rng, batch=None):
    """Draw n N(0,1) variates by acceptance-rejection with a Cauchy envelope."""
    if batch is None:
        batch = max(1_000, int(n * C * 1.2))    # expect ~C proposals per acceptance

    ys, keeps, n_acc = [], [], 0
    while n_acc < n:
        y = sample_cauchy(batch, rng)           # Step 1: Y ~ g   (Q1 sampler)
        u = rng.random(batch)                   # Step 2: U ~ U(0,1)
        keep = accept_mask(y, u, C)             # Step 3: accept?
        ys.append(y); keeps.append(keep)
        n_acc += int(keep.sum())

    return acceptance_tracker(np.concatenate(ys), np.concatenate(keeps), n)

In [ ]:
# ==================================================================
# Q2 driver -- given.  Do not edit.
# ==================================================================
rng = fresh_rng()
z, info = normal_from_cauchy(N, C, rng)

# C too small => f/(C*g) can exceed 1 => the method is INVALID.
worst = ratio(np.linspace(-10, 10, 20_001)).max() / C
if worst > 1 + 1e-9:
    print(f"  WARNING: f/(C*g) reaches {worst:.4f} > 1.  C = {C} is not an upper "
          f"bound; these samples are BIASED.\n")

assert len(z) == N, f"expected {N} samples, got {len(z)}"

print(f"C  (your envelope constant)   = {C:.6f}")
print(f"theoretical acceptance 1/C    = {1 / C:.4f}")
print(f"empirical   acceptance        = {info['acceptance_rate']:.4f}")
print(f"proposals used for {N:,} draws = {info['n_proposals']:,}")
print(f"mean proposals per acceptance = {info['trials'].mean():.4f}   "
      f"(theory {C:.4f})")

In [ ]:
# ==================================================================
# Q2 diagnostics -- given.  Do not edit.
# ==================================================================
check_continuous(z, stats.norm(), "Normal(0,1) via acceptance-rejection",
                 xlim=(-4.5, 4.5), bins=70)

print(f"sample mean     = {z.mean():+.4f}   (theory 0)")
print(f"sample sd       = {z.std(ddof=1):.4f}   (theory 1)")
print(f"sample skewness = {stats.skew(z):+.4f}   (theory 0)")

In [ ]:
# ==================================================================
# Q2 picture -- given.  Do not edit.
#
# Left : where the wasted proposals go.  The shaded gap between f and C*g is
#        the rejection region; a proposal landing there is thrown away.
# Right: proposals needed per acceptance should be Geometric(1/C), mean C.
# ==================================================================
rng_pic = np.random.default_rng(SEED + 1)
n_show = 4000
Y = sample_cauchy(n_show, rng_pic)
U = rng_pic.random(n_show)
acc = accept_mask(Y, U, C)
height = U * C * g_cauchy(Y)

xs = np.linspace(-6, 6, 500)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))

ax[0].fill_between(xs, f_normal(xs), C * g_cauchy(xs), color=ORANGE, alpha=0.15)
ax[0].plot(Y[acc], height[acc], ".", ms=2, color=GREEN,
           label=f"accept ({acc.mean():.1%})")
ax[0].plot(Y[~acc], height[~acc], ".", ms=2, color=RED,
           label=f"reject ({1 - acc.mean():.1%})")
ax[0].plot(xs, f_normal(xs), color=BLUE, lw=2.2, label="target $f$")
ax[0].plot(xs, C * g_cauchy(xs), color=ORANGE, lw=2.2, label="envelope $C g$")
ax[0].set_xlim(-6, 6); ax[0].set_ylim(0, 0.55)
ax[0].set_xlabel("x"); ax[0].set_title("Simulations under the envelope")
ax[0].legend(fontsize=7, ncol=2)

kk = np.arange(1, 11)
trials = info["trials"]
ax[1].bar(kk - 0.2, [(trials == k).mean() for k in kk], width=0.4, color=BLUE,
          label="observed")
ax[1].bar(kk + 0.2, stats.geom.pmf(kk, 1 / C), width=0.4, color=ORANGE,
          label="Geometric(1/C)")
ax[1].set_xlabel("proposals needed for one acceptance")
ax[1].set_title(f"mean = {trials.mean():.3f}   (theory {C:.3f})")
ax[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

In [ ]:
# ==================================================================
# FINAL SUMMARY -- given.  Do not edit.
# Read these numbers out at the viva.
# ==================================================================
print("=" * 58)
print(f"  SEED                            {SEED}")
print(f"  Q1  round-trip error            {err_roundtrip:.2e}")
print(f"  Q1  median                      {np.median(x_cauchy):+.4f}")
print(f"  Q2  ratio error                 {err_ratio:.2e}")
print(f"  Q2  C                           {C:.6f}")
print(f"  Q2  empirical acceptance        {info['acceptance_rate']:.4f}")
print(f"  Q2  proposals for {N:,}       {info['n_proposals']:,}")
print(f"  Q2  mean                        {z.mean():+.4f}")
print(f"  Q2  sd                          {z.std(ddof=1):.4f}")
print("=" * 58)

---
### Before you submit

* Run **Kernel → Restart & Run All** and confirm every cell executes top to bottom without error.
* Check that no `NotImplementedError` remains and that all plots are visible.
* Confirm both self-check cells print `PASS`.
* Save the notebook and keep it open - you will be asked to explain your derivations for Q1(b) and Q2(b).